- We'll start with building and training the model described in the seminal 2020 paper Denoising Diffusion Probabilistic Models [(DDPM)](https://arxiv.org/abs/2006.11239).
- For more context, while diffusion models were technically invented [back in 2015](https://arxiv.org/abs/1503.03585), diffusion models flew under the radar until this 2020 paper since they were complicated and difficult to train.
- The 2020 paper introducing DDPMs made some crucial assumptions that significantly simplify the model training and generation processes, as we will see here.
- Later versions of diffusion models all build upon the same framework introduced in this paper.

# Load deps

In [22]:
# # # if src modules imported
# # from google.colab import drive
# # drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# # app_path = '/home/amin/Projects/fast-ai/'
# sys.path.append(app_path)

In [ ]:
# !pip install -q torcheval diffusers ffmpeg
# !uv add torcheval diffusers ffmpeg

In [ ]:
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from functools import partial
import torchvision.transforms.functional as TF
from torch import nn,optim
from torch.optim import lr_scheduler
from datasets import load_dataset
from diffusers import UNet2DModel

from src.utils import set_seed
from src.datasets import inplace, DataLoaders
from src.learner import Learner, TrainCB, DeviceCB, ProgressCB, MetricsCB
from src.sgd import BatchSchedCB
from src.datasets import show_image, show_images
from src.conv import def_device

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
set_seed(42)
mpl.rcParams['image.cmap'] = 'gray_r'
ds_name = "fashion_mnist"
ds_xl,ds_yl = 'image','label'
bs = 128
num_dl_workers = 2
lr = 4e-3
epochs = 2

# Load data

- To make life simpler (mostly with the model architecture), we'll resize the 28x28 images to 32x32
- Note that while we do get the labels for the dataset, we actuallydon't care about that for our task of **unconditional** image generation.

In [ ]:
@inplace
def transformi(b): b[ds_xl] = [TF.resize(TF.to_tensor(o), (32,32)) for o in b[ds_xl]]
# TODO: why didn't we normalize the input?

dsd = load_dataset(ds_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

In [ ]:
dt = dls.train
xb,yb = next(iter(dt))
print(xb.shape)
print(yb[:10])
show_images(xb[:16], figsize=(5,5));

# U-net model

- A U-net looks something like this:

<img src="https://huggingface.co/blog/assets/78_annotated-diffusion/unet_architecture.jpg" width="600">

- The DDPM U-net is a modification of this with some modern tricks like using attention.
- We will cover how U-nets are created and how modules like attention work in future lessons.
- For now, we'll import the U-net from the diffusers library:

In [ ]:
model = UNet2DModel(in_channels=1, out_channels=1, block_out_channels=(32, 64, 128, 128))

# Training using callback!

DDPM is trained quite simply in a few steps:
1. randomly select some timesteps in an iterative noising process.
2. Add noise corresponding to this timestep to the original image. For increasing timesteps, the variance of the noise increases.
3. Pass in this noisy image and the timestep to our model
4. Model is trained with an MSE loss between the model output and the amount of noise added to the image


We will implement this in a callback. The callback will randomly select the timestep and create the noisy image before setting up our input and ground truth tensors for the model forward pass and loss calculation.

After training, we need to sample from this model. This is an iterative denoising process starting from pure noise. We simply keep removing noise predicted by the neural network, but we do it with an expected noise schedule that is reverse of what we saw during training. This is also done in our callback.

In [ ]:
class DDPMCB(TrainCB):
    order = DeviceCB.order+1
    def __init__(self, n_steps, beta_min, beta_max):
        super().__init__()
        self.n_steps,self.βmin,self.βmax = n_steps,beta_min,beta_max
        # variance schedule, linearly increased with timestep
        self.β = torch.linspace(self.βmin, self.βmax, self.n_steps)
        self.α = 1. - self.β 
        self.ᾱ = torch.cumprod(self.α, dim=0)
        self.σ = self.β.sqrt()

    def predict(self, learn): learn.preds = learn.model(*learn.batch[0]).sample
    
    def before_batch(self, learn):
        # TODO: we need to get/set device only once not for each batch
        device = learn.batch[0].device
        ε = torch.randn(learn.batch[0].shape, device=device)  # noise, x_T
        x0 = learn.batch[0] # original images, x_0
        self.ᾱ = self.ᾱ.to(device)
        n = x0.shape[0]
        # select random timesteps
        t = torch.randint(0, self.n_steps, (n,), device=device, dtype=torch.long)
        ᾱ_t = self.ᾱ[t].reshape(-1, 1, 1, 1).to(device)
        xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε #noisify the image
        # input to our model is noisy image and timestep, ground truth is the noise
        # TODO: is the target the totla noise (below) or just the added/scaled noise??
        learn.batch = ((xt, t), ε)
    
    @torch.no_grad()
    def sample(self, learn, sz):
        device = next(learn.model.parameters()).device
        x_t = torch.randn(sz, device=device)
        preds = []
        for t in reversed(range(self.n_steps)):
            t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
            z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(device)
            ᾱ_t1 = self.ᾱ[t-1]  if t > 0 else torch.tensor(1)
            b̄_t = 1 - self.ᾱ[t]
            b̄_t1 = 1 - ᾱ_t1
            noise_pred = learn.model(x_t, t_batch).sample
            x_0_hat = ((x_t - b̄_t.sqrt() * noise_pred)/self.ᾱ[t].sqrt()).clamp(-1,1)
            x0_coeff = ᾱ_t1.sqrt()*(1-self.α[t])/b̄_t
            xt_coeff = self.α[t].sqrt()*b̄_t1/b̄_t
            x_t = x_0_hat*x0_coeff + x_t*xt_coeff + self.σ[t]*z
            preds.append(x_t.cpu())
        return preds

## Demonstrating a single batch logic

In [ ]:
# from src.learner import SingleBatchCB, CompletionCB

# ddpm_cb = DDPMCB(n_steps=1000, beta_min=0.0001, beta_max=0.02)
# cbs = [ddpm_cb, DeviceCB(), SingleBatchCB(), CompletionCB()]
# learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=optim.Adam)
# learn.fit(n_epochs=1, valid=False)

In [ ]:
# print(type(learn.batch))
# print(learn.batch[0][0].shape)
# print(learn.batch[0][1].shape)
# print(learn.batch[1].shape)
# print("=============")
# model_fwd_out = learn.model(*learn.batch[0])
# print(type(model_fwd_out))
# print(model_fwd_out.__dict__.keys())
# print(model_fwd_out.sample.shape)
# print("=============")
# eps = learn.batch[1]
# predicted_eps = model_fwd_out.sample
# xt = learn.batch[0][0]
# print(eps.mean((-2,-1)).flatten())
# print(predicted_eps.mean((-2,-1)).flatten())
# print(xt.mean((-2,-1)).flatten())

## Main train loop

Okay now we're ready to train a model!

Let's create our `Learner`. We'll add our callbacks and train with MSE loss.

We specify the number of timesteps and the minimum and maximum variance for the DDPM model.

In [ ]:
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
ddpm_cb = DDPMCB(n_steps=1000, beta_min=0.0001, beta_max=0.02)
cbs = [ddpm_cb, DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched)]
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=optim.Adam)
# learn.fit(epochs)

## Save and Load model

In [ ]:
mdl_path = Path('models')
# mdl_path = Path('/home/amin/Projects/fast-ai/models')
mdl_path.mkdir(exist_ok=True)

# torch.save(learn.model, mdl_path/'fashion_ddpm.pkl')

In [ ]:
learn.model = torch.load(
    mdl_path/'fashion_ddpm.pkl',
    weights_only=False,
    map_location=torch.device(def_device)
)

# Inference

In [ ]:
print(model.config.center_input_sample)
# we didn't scale the images.
# no need for any reverse transform to display images

In [ ]:
set_seed(42)
samples = ddpm_cb.sample(learn, (16, 1, 32, 32))
print(len(samples)) # it returns samples for all steps
show_images(samples[-1], figsize=(5,5));

In [ ]:
# import pickle
# # with open("samp.pkl", "wb") as file: pickle.dump(file=file, obj=samples)
# with open("samp.pkl", "rb") as file: samples = pickle.load(file=file)

In [ ]:
%matplotlib inline
import matplotlib.animation as animation
from IPython.display import HTML

fig,ax = plt.subplots(figsize=(3,3))
def _show_i(i): return show_image(-samples[i][9], ax=ax, animated=True).get_images()
r = list(range(800,990, 5))+list(range(990,1000))+[999]*10
ims = [_show_i(i) for i in r]
animate = animation.ArtistAnimation(fig, ims, interval=50, blit=True, repeat_delay=3000)
HTML(animate.to_html5_video())

**Note** that I only take the steps between 800 and 1000 since most of the previous steps are actually quite noisy.
- This is a limitation of the noise schedule used for small images
    - this means that we're not making full use of those 800 steps!!!
- papers like [Improved DDPM](https://arxiv.org/abs/2102.09672) suggest other noise schedules for this purpose
- TODO: try out the noise schedule from Improved DDPM and see if it helps.
    - Does the improvement depend on the image size?